# Algoritmos de ordenamiento avanzados (parte 02)

En esta sesión nuestro objetivo es analizar, implementar y evaluar el algortimo Heap Sort (Ordenamiento por Montículos).

El estudio de este algoritmo nos permite explorar otro paradigma de desarrollo de algoritmos:

1. ***Ordenamiento por comparación***: Sujeto al límite inferior teórico de $\Omega(n \log n)$. Heap Sort representa un diseño óptimo en espacio ($O(1)$ auxiliar) y tiempo ($O(n \log n)$ en el peor de los casos).

In [1]:
#include <iostream>
#include <vector>
#include <string>

/**
 * @brief Imprime colecciones iterables. Compatible con C++17 para entornos REPL/Jupyter.
 * Utiliza tipado genérico (Templates) en lugar de Concepts para evitar caídas del kernel.
 * @tparam Container Tipo del contenedor iterativo.
 * @param collection El contenedor a imprimir.
 * @param label Etiqueta descriptiva para la salida estándar.
 */
template <typename Container>
void print_collection(const Container& collection, const std::string& label = "Estado de la colección") {
    std::cout << label << ": [ ";
    for (const auto& elem : collection) {
        std::cout << elem << " ";
    }
    std::cout << "]\n";
}



void ejemplo_01() {
    std::vector<double> initial_data = {0.78, 0.17, 0.39, 0.26, 0.72, 0.94, 0.21, 0.12, 0.23, 0.68};
    print_collection(initial_data, "Datos iniciales");
}

ejemplo_01();

Datos iniciales: [ 0.78 0.17 0.39 0.26 0.72 0.94 0.21 0.12 0.23 0.68 ]


# Heapsort

Hasta este punto en la teoría algorítmica estándar, usualmente nos hemos apoyado en algoritmos de ordenamiento eficientes en la práctica como Merge Sort o Quick Sort. Sin embargo, ambos presentan compromisos técnicos (trade-offs) severos en escenarios críticos:

1. ***Límites Espaciales***: Merge Sort requiere memoria dinámica adicional $\Theta(n)$. En sistemas embebidos, de tiempo real o al procesar conjuntos de datos masivos, la latencia de asignación de memoria (allocation) y la presión sobre la caché del CPU hacen prohibitivo este costo de espacio.

2. ***Degradación del Tiempo***: Quick Sort, a pesar de su buen rendimiento promedio, está sujeto a un peor caso teórico de $\Theta(n^2)$ si la selección de pivotes es deficiente.

3. ***Heap Sort*** surge como una arquitectura algorítmica óptima que combina los mejores atributos de sus predecesores: garantiza un tiempo asintótico de $O(n \log n)$ en el peor de los casos y opera estrictamente in-place (complejidad espacial $O(1)$ auxiliar). Esto lo logra estructurando un arreglo subyacente de manera que simule un árbol binario casi completo

In [2]:
#include <chrono>
#include <random>

/**
 * @brief Demostración del impacto de asignación de memoria.
 * Evidencia por qué Heap Sort (espacio auxiliar O(1)) es superior en
 * entornos de memoria restringida frente a algoritmos con espacio O(n).
 */
void memory_overhead_motivation() {
    const size_t N = 50000000; // 50 Millones de registros
    std::vector<int> data(N);
    
    // Generación de ruido estadístico para el dataset
    std::mt19937 gen(42);
    std::uniform_int_distribution<int> dist(1, 100000);
    for(auto& x : data) x = dist(gen);

    // Medición de latencia introducida puramente por la clonación de memoria O(n)
    auto start_alloc = std::chrono::high_resolution_clock::now();
    
    // Simula el requerimiento de buffer auxiliar (e.g., Merge Sort)
    std::vector<int> auxiliary_buffer = data; 
    
    auto end_alloc = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> alloc_time = end_alloc - start_alloc;
    
    std::cout << "Latencia de asignacion/copia de memoria O(n): " 
              << alloc_time.count() << " ms\n";
              
    std::cout << "Heap Sort evitara esta asignacion operando directamente sobre el arreglo base.\n";
}

memory_overhead_motivation();


Latencia de asignacion/copia de memoria O(n): 65.7869 ms
Heap Sort evitara esta asignacion operando directamente sobre el arreglo base.


* La línea `std::vector<int> auxiliary_buffer = data;` demuestra un costo invisible en la complejidad teórica clásica: la clonación de Heap Memory (la zona de memoria RAM del SO). Reservar grandes bloques continuos demanda ciclos de CPU al memory allocator del SO y contamina las líneas de la caché L1/L2. Heap Sort anula esto mediante manipulaciones de índices in-place.

* Relevancia del Estándar: La propia Standard Template Library (STL) de C++ implementa `std::sort` usando el algoritmo IntroSort (un híbrido). Este inicia ejecutando Quick Sort, pero monitorea la profundidad de la pila de llamadas; si la recursión excede $2 \log_2(n)$, transiciona forzosamente a Heap Sort para asegurar un blindaje contra la degradación a $O(n^2)$. 

## Implementación

* ***MAX-HEAPIFY***: Esta rutina asume que los subárboles izquierdo y derecho del nodo $i$ ya son max-heaps, pero que $A[i]$ podría ser menor que sus hijos. Al hacer flotar el valor hacia abajo, el tiempo de ejecución está acotado por la altura del árbol. Su complejidad temporal es de $O(\log n)$ (Cormen, pág. 164).

* ***BUILD-MAX-HEAP***: Recorre los nodos internos del árbol de abajo hacia arriba (desde $\lfloor n/2 \rfloor$ hasta $1$). Aunque podría parecer que cuesta $O(n \log n)$, un análisis riguroso (usando series geométricas) demuestra que la mayor parte de los nodos están cerca de las hojas y la altura a recorrer es mínima. Por lo tanto, el costo asintótico real es estrictamente $O(n)$ lineal (Cormen, pág. 167).

* ***HEAPSORT***: El algoritmo primero construye el heap en $O(n)$. Luego extrae el elemento máximo (la raíz $A[1]$), lo coloca al final del arreglo, reduce el tamaño del heap y llama a MAX-HEAPIFY. Este proceso de extracción ocurre $n-1$ veces, tomando $O(\log n)$ cada uno. Complejidad total: $O(n \log n)$ garantizado en el peor caso (Cormen, pág. 170). Espacio auxiliar: $O(1)$.


* ***Transición 1-Index a 0-Index***: En matemáticas es conveniente indexar desde 1, por lo que $Left(i) = 2i$. En C++, un array empieza en 0. Por ende, la aritmética sufre un shift: $Left(i) = 2i + 1$ y $Right(i) = 2i + 2$.

* ***Uso de Heap Lógico (heap_size)***: Note que el tamaño del `std::vector` jamás cambia. Mutamos una variable escalar `heap_size` que impone un límite "lógico" para que `max_heapify` ignore los valores máximos que ya han sido transportados al final del arreglo, garantizando la complejidad espacial $O(1)$.

* ***Conversión Estricta de Tipos***: Los iteradores para `build_max_heap` comienzan con un tipo int con signo. Esto es crítico: si se usara `size_t` (sin signo), la condición `i >= 0` causaría un Underflow al llegar a cero, originando un bucle infinito (comportamiento indefinido). Este es un error clásico en implementaciones C++ que evalúan heurísticas matemáticas de forma directa.


```cpp
// Adaptación de los macros de Cormen al esquema basado en índice 0 de C++ (Mahdi, Cap 9.2)
inline size_t parent(size_t i) { return (i - 1) / 2; }
inline size_t left(size_t i) { return 2 * i + 1; }
inline size_t right(size_t i) { return 2 * i + 2; }

/**
 * Mantiene la propiedad de Max-Heap en el subárbol con raíz en 'i'.
 * Costo: O(log n)
 */
template <typename T>
void max_heapify(std::vector<T>& A, size_t heap_size, size_t i) {
    size_t l = left(i);
    size_t r = right(i);
    size_t largest = i;

    // Verificar si el hijo izquierdo es mayor que la raíz actual
    if (l < heap_size && A[l] > A[i]) {
        largest = l;
    }
    
    // Verificar si el hijo derecho es mayor que el 'largest' actual
    if (r < heap_size && A[r] > A[largest]) {
        largest = r;
    }

    // Si el más grande no es la raíz, intercambiamos y aplicamos recursión
    if (largest != i) {
        std::swap(A[i], A[largest]);
        // En código de producción estricto, esto puede optimizarse a iterativo
        // para evitar el uso del Call Stack, pero mantenemos la recursión pedagógica.
        max_heapify(A, heap_size, largest);
    }
}

/**
 * @brief Construye un Max-Heap bottom-up a partir de un arreglo desordenado.
 * Costo: O(n)
 */
template <typename T>
void build_max_heap(std::vector<T>& A) {
    size_t heap_size = A.size();
    // Empezamos desde el último nodo interno hacia la raíz
    for (int i = (heap_size / 2) - 1; i >= 0; --i) {
        max_heapify(A, heap_size, static_cast<size_t>(i));
    }
}

/**
 * @brief Rutina principal de Heap Sort.
 * Costo: O(n log n) In-Place
 */
template <typename T>
void heap_sort(std::vector<T>& A) {
    build_max_heap(A);
    size_t heap_size = A.size();
    
    // Extraer elementos uno a uno desde el montículo
    for (int i = A.size() - 1; i > 0; --i) {
        // Movemos la raíz (el máximo actual) al final de nuestro arreglo particionado
        std::swap(A[0], A[i]);
        heap_size--; // Reducimos el tamaño lógico del heap
        
        // Restauramos la propiedad de Max-Heap en la nueva raíz
        max_heapify(A, heap_size, 0);
    }
}

# Análisis comparativo de algoritmos de ordenamiento

In [3]:
#include <iostream>
#include <vector>
#include <algorithm>
#include <chrono>
#include <random>
#include <iomanip>


// ============================================================================
// 1. IMPLEMENTACIÓN MANUAL ESTRICTA (Basada en Cormen)
// ============================================================================

namespace Manual {
    // Adaptación de los macros de Cormen al esquema basado en índice 0 de C++ (Mahdi, Cap 9.2)
    inline size_t parent(size_t i) { return (i - 1) / 2; }
    inline size_t left(size_t i) { return 2 * i + 1; }
    inline size_t right(size_t i) { return 2 * i + 2; }

    /**
     * Mantiene la propiedad de Max-Heap en el subárbol con raíz en 'i'.
     * Costo: O(log n)
     */
    template <typename T>
    void max_heapify(std::vector<T>& A, size_t heap_size, size_t i) {
        size_t l = left(i);
        size_t r = right(i);
        size_t largest = i;

        // Verificar si el hijo izquierdo es mayor que la raíz actual
        if (l < heap_size && A[l] > A[i]) {
            largest = l;
        }
        
        // Verificar si el hijo derecho es mayor que el 'largest' actual
        if (r < heap_size && A[r] > A[largest]) {
            largest = r;
        }

        // Si el más grande no es la raíz, intercambiamos y aplicamos recursión
        if (largest != i) {
            std::swap(A[i], A[largest]);
            // En código de producción estricto, esto puede optimizarse a iterativo
            // para evitar el uso del Call Stack, pero mantenemos la recursión pedagógica.
            max_heapify(A, heap_size, largest);
        }
    }

    /**
     * Construye un Max-Heap bottom-up a partir de un arreglo desordenado.
     * Costo: O(n)
     */
    template <typename T>
    void build_max_heap(std::vector<T>& A) {
        size_t heap_size = A.size();
        // Empezamos desde el último nodo interno hacia la raíz
        for (int i = (heap_size / 2) - 1; i >= 0; --i) {
            max_heapify(A, heap_size, static_cast<size_t>(i));
        }
    }

    /**
     * Rutina principal de Heap Sort.
     * Costo: O(n log n) In-Place
     */
    template <typename T>
    void heap_sort(std::vector<T>& A) {
        build_max_heap(A);
        size_t heap_size = A.size();
        
        // Extraer elementos uno a uno desde el montículo
        for (int i = A.size() - 1; i > 0; --i) {
            // Movemos la raíz (el máximo actual) al final de nuestro arreglo particionado
            std::swap(A[0], A[i]);
            heap_size--; // Reducimos el tamaño lógico del heap
            
            // Restauramos la propiedad de Max-Heap en la nueva raíz
            max_heapify(A, heap_size, 0);
        }
    }

    // Implementación clásica de Insertion Sort O(n^2)
    template <typename T>
    void insertion_sort(std::vector<T>& A) {
        for (size_t i = 1; i < A.size(); ++i) {
            T key = A[i];
            int j = i - 1;
            while (j >= 0 && A[j] > key) {
                A[j + 1] = A[j];
                j = j - 1;
            }
            A[j + 1] = key;
        }
    }
}

// ============================================================================
// UTILIDAD DE MEDICIÓN DE ALTA PRECISIÓN
// ============================================================================

struct Timer {
    std::chrono::high_resolution_clock::time_point start;
    Timer() : start(std::chrono::high_resolution_clock::now()) {}
    
    double elapsed_ms() const {
        auto end = std::chrono::high_resolution_clock::now();
        return std::chrono::duration<double, std::milli>(end - start).count();
    }
};

// ============================================================================
// LABORATORIO DE ESTRÉS EMPÍRICO
// ============================================================================

void run_comprehensive_benchmark() {
    const size_t N_MASSIVE = 10'000'000; // 10 Millones para O(n log n)
    const size_t N_SMALL   = 50'000;     // 50 Mil para O(n^2)
    
    std::cout << "========================================================\n";
    std::cout << " BENCHMARK MULTI-PARADIGMA (N = " << N_MASSIVE << ")\n";
    std::cout << "========================================================\n\n";

    // Generación de la muestra aleatoria
    std::cout << "[*] Generando " << N_MASSIVE << " registros flotantes... ";
    Timer t_gen;
    std::mt19937_64 gen(42); 
    std::uniform_real_distribution<double> dist(0.0, 1000000.0);
    
    std::vector<double> base_data(N_MASSIVE);
    for (auto& val : base_data) val = dist(gen);
    std::cout << std::fixed << std::setprecision(2) << t_gen.elapsed_ms() << " ms.\n\n";

    // Clonación estricta
    std::vector<double> data_insertion(base_data.begin(), base_data.begin() + N_SMALL); // Solo 50k
    std::vector<double> data_manual_heap = base_data;
    std::vector<double> data_stl_heap    = base_data;
    std::vector<double> data_quick       = base_data;
    std::vector<double> data_merge       = base_data;

    // --- PRUEBA 0: INSERTION SORT (O(n^2) - Submuestra) ---
    std::cout << "[>] Ejecutando Insertion Sort (Solo " << N_SMALL << " elementos)... \n";
    Timer t_ins;
    Manual::insertion_sort(data_insertion);
    double ms_ins = t_ins.elapsed_ms();
    std::cout << "    Tiempo: " << ms_ins << " ms.\n";
    std::cout << "    (Proyeccion para 10M: ~" << (ms_ins * 40000.0) / 1000.0 / 3600.0 << " horas!)\n\n";

    // --- PRUEBA 1: MANUAL HEAP SORT (O(n log n) in-place) ---
    std::cout << "[>] Ejecutando Manual Heap Sort (" << N_MASSIVE << ")... \n";
    Timer t_manual;
    Manual::heap_sort(data_manual_heap);
    double ms_manual = t_manual.elapsed_ms();
    std::cout << "    Tiempo: " << ms_manual << " ms.\n\n";

    // --- PRUEBA 2: STL HEAP SORT (O(n log n) in-place / Sift-Down) ---
    std::cout << "[>] Ejecutando STL Heap Sort (" << N_MASSIVE << ")... \n";
    Timer t_stl;
    std::make_heap(data_stl_heap.begin(), data_stl_heap.end());
    std::sort_heap(data_stl_heap.begin(), data_stl_heap.end());
    double ms_stl = t_stl.elapsed_ms();
    std::cout << "    Tiempo: " << ms_stl << " ms.\n\n";

    // --- PRUEBA 3: MERGE SORT (O(n log n) - O(n) memoria extra) ---
    std::cout << "[>] Ejecutando STL Merge Sort / Stable Sort (" << N_MASSIVE << ")... \n";
    Timer t_merge;
    std::stable_sort(data_merge.begin(), data_merge.end());
    double ms_merge = t_merge.elapsed_ms();
    std::cout << "    Tiempo: " << ms_merge << " ms.\n\n";

    // --- PRUEBA 4: QUICK SORT (O(n log n) - IntroSort optimizado) ---
    std::cout << "[>] Ejecutando STL Quick Sort / IntroSort (" << N_MASSIVE << ")... \n";
    Timer t_quick;
    std::sort(data_quick.begin(), data_quick.end());
    double ms_quick = t_quick.elapsed_ms();
    std::cout << "    Tiempo: " << ms_quick << " ms.\n\n";

    // --- CONCLUSIONES ARQUITECTONICAS ---
    std::cout << "========================================================\n";
    std::cout << " RESUMEN DE LATENCIAS PARA " << N_MASSIVE << " ELEMENTOS:\n";
    std::cout << " 1. STL Quick Sort  : " << ms_quick  << " ms (El mas rapido - Cache-Friendly)\n";
    std::cout << " 2. STL Merge Sort  : " << ms_merge  << " ms (Costo oculto de malloc O(n))\n";
    std::cout << " 3. STL Heap Sort   : " << ms_stl    << " ms (Lento por cache misses, pero O(1) memoria)\n";
    std::cout << " 4. Manual Heap Sort: " << ms_manual << " ms (Overhead por std::swap vs Sift-Down)\n";
    std::cout << "========================================================\n";
}

run_comprehensive_benchmark();

 BENCHMARK MULTI-PARADIGMA (N = 10000000)

[*] Generando 10000000 registros flotantes... 9016.97 ms.

[>] Ejecutando Insertion Sort (Solo 50000 elementos)... 
    Tiempo: 4443.50 ms.
    (Proyeccion para 10M: ~49.37 horas!)

[>] Ejecutando Manual Heap Sort (10000000)... 
    Tiempo: 13164.36 ms.

[>] Ejecutando STL Heap Sort (10000000)... 
    Tiempo: 16086.70 ms.

[>] Ejecutando STL Merge Sort / Stable Sort (10000000)... 
    Tiempo: 4094.86 ms.

[>] Ejecutando STL Quick Sort / IntroSort (10000000)... 
    Tiempo: 3782.28 ms.

 RESUMEN DE LATENCIAS PARA 10000000 ELEMENTOS:
 1. STL Quick Sort  : 3782.28 ms (El mas rapido - Cache-Friendly)
 2. STL Merge Sort  : 4094.86 ms (Costo oculto de malloc O(n))
 3. STL Heap Sort   : 16086.70 ms (Lento por cache misses, pero O(1) memoria)
 4. Manual Heap Sort: 13164.36 ms (Overhead por std::swap vs Sift-Down)
